# Stage-1 Interpreter — slice-first de-risk run (caption → corpus → train → score)

Runs the CHEAP de-risk before committing the full caption budget: caption ~500 LUTs, build the
leakage-safe interpreter corpus, full-FT **Qwen2.5-0.5B-Instruct** (minutes on a T4 — does NOT need
the A100 the collapse-fix loop is using), and score on the `split_unit_id` holdout.

**GO/NO-GO:** the interpreter must beat a trivial always-`grade` baseline on `attribute_f1` +
route accuracy. If a 0.5B model on ~2.5k rows can't, stop before the full run (~2h + pennies spent,
not 15h + A100). CELL 1 prompts for `TFY_API_KEY` (teacher) + `HF_TOKEN` (read); the base URL is
pre-filled — press Enter to keep any value, or type a new one to fix a wrong entry.

In [1]:
# CELL 1 - provision (clone, install, tokens, stage corpus for active_rows.jsonl)
import os, pathlib, subprocess, sys
REPO, BRANCH = '/content/SLM', 'feat/two-stage'
if not pathlib.Path(REPO, '.git').is_dir():
    subprocess.run(['git', 'clone', 'https://github.com/ericrcwu001/SLM', REPO], check=True)
os.chdir(REPO)
subprocess.run(['git', 'fetch', 'origin', BRANCH], check=True)
subprocess.run(['git', 'checkout', BRANCH], check=True)
subprocess.run(['git', 'reset', '--hard', f'origin/{BRANCH}'], check=True)
if not os.environ.get('SLM_PROVISIONED'):
    # [frontier] brings the openai SDK the teacher gateway needs; [sft] covers torch/transformers.
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[sft,frontier]'], check=True)
    os.environ['SLM_PROVISIONED'] = '1'
print('HEAD:', subprocess.run(['git','log','--oneline','-1'], capture_output=True, text=True).stdout.strip())

def _env_token(name):
    for _p in ('/content/SLM/.env', '/content/.env', '.env'):
        fp = pathlib.Path(_p)
        if fp.is_file():
            for _l in fp.read_text().splitlines():
                if _l.strip().startswith(name + '='):
                    return _l.split('=', 1)[1].strip().strip('"').strip("'")
    return None

# ALWAYS re-prompt (so a wrong/missing key is trivial to fix). Press Enter to KEEP the current value
# (from the environment or an uploaded .env); type a new value to overwrite it. TFY_BASE_URL is
# pre-filled with the known gateway.
import getpass
_DEFAULTS = {'TFY_BASE_URL': 'https://tfy.promptlens.trilogy.com/v1'}
for _n in ('HF_TOKEN', 'TFY_BASE_URL', 'TFY_API_KEY'):
    _cur = os.environ.get(_n) or _env_token(_n) or _DEFAULTS.get(_n, '')
    if not _cur:
        _hint = ' [required]'
    elif _n == 'TFY_BASE_URL':
        _hint = f' [Enter=keep {_cur}]'          # a URL is not secret; show it in full
    else:
        _hint = f' [Enter=keep ...{_cur[-4:]}]'  # token: show only the last 4 chars
    os.environ[_n] = getpass.getpass(f'{_n}{_hint}: ').strip() or _cur
# Honest readiness: BOTH TFY vars must be present (the old check only tested the base URL, which
# masked a missing TFY_API_KEY and let the caption cell fail with a hand-off).
print('HF_TOKEN set:', bool(os.environ.get('HF_TOKEN')),
      '| TFY ready:', bool(os.environ.get('TFY_BASE_URL')) and bool(os.environ.get('TFY_API_KEY')))

# The interpreter needs active_rows.jsonl (the out_of_scope refuse rows + split_unit_id source).
# We caption text-only (--no-image), so the full image/VQ corpus is not required; stage only if the
# jsonl is missing.
os.environ['SLM_ARTIFACT_ROOT'] = '/content/slm'
if not pathlib.Path('data/active_sft/active_rows.jsonl').is_file():
    subprocess.run(['slm_stage', 'stage', '--durable-root', 'hf://datasets/ericrcwu/LUT_SLM',
                    '--local-root', '/content/slm'], check=True)
print('active_rows.jsonl:', pathlib.Path('data/active_sft/active_rows.jsonl').is_file())

HEAD: 42ae783 Add Stage-1 interpreter pipeline: caption corpus + interpreter SFT
HF_TOKEN set: True | TFY ready: True
active_rows.jsonl: True


In [2]:
# CELL 2 - data: caption slice + clarify/out_of_gamut supplement + unified leakage-safe corpus
import subprocess, sys
LIMIT = 500          # slice size (LUTs). Full run later drops this.
# 2a. grade captions (text-only teacher; --workers for speed).
rc = subprocess.run([sys.executable, '-m', 'scripts.generate_captions',
                     '--limit', str(LIMIT), '--no-image', '--workers', '6']).returncode
assert rc == 0, 'caption gen failed / handed off (check TFY_BASE_URL + TFY_API_KEY)'
# 2b. clarify + out_of_gamut route supplement (full 3-way).
rc = subprocess.run([sys.executable, '-m', 'scripts.generate_route_supplement',
                     '--n-clarify', '150', '--n-gamut', '150']).returncode
assert rc == 0, 'route supplement failed / handed off'
# 2c. unify + stamp split_unit_id (the leakage fix).
rc = subprocess.run([sys.executable, '-m', 'scripts.build_interpreter_corpus']).returncode
assert rc == 0, 'corpus build failed'
import json
print(json.dumps(json.load(open('data/interpreter/interpreter_corpus_manifest.json')), indent=2))

{
  "interpreter_corpus_version": "interpreter_v1",
  "out": "data/interpreter/interpreter_rows.jsonl",
  "sources": {
    "caption_rows": "data/active_sft/caption_rows.jsonl",
    "active_rows": "data/active_sft/active_rows.jsonl",
    "route_supplement": "data/active_sft/route_supplement_rows.jsonl"
  },
  "total_rows": 2924,
  "dropped_missing_unit": 0,
  "fallback_key_count": 0,
  "by_route": {
    "grade": 2500,
    "refuse": 422,
    "clarify": 2
  },
  "by_refuse_kind": {
    "out_of_scope": 272,
    "out_of_gamut": 150
  },
  "by_source_family": {
    "caption": 2500,
    "unsupported_teacher": 424
  },
  "by_style": {
    "literal": 500,
    "metaphor": 500,
    "mood": 500,
    "concept": 500,
    "slang": 500
  },
  "distinct_units": 707,
  "units_per_route": {
    "grade": 283,
    "clarify": 2,
    "refuse": 422
  }
}


In [12]:
!cd /content/SLM && git fetch origin feat/two-stage -q && git reset --hard origin/feat/two-stage -q && git log --oneline -1

09fab4a (HEAD -> feat/two-stage, origin/feat/two-stage) Interpreter slice: drop non-grade upsample to 1x, epochs 3->5


In [13]:
# CELL 3 - train the interpreter (full-FT Qwen2.5-0.5B-Instruct; minutes on a T4)
import subprocess, sys, glob
rc = subprocess.run([sys.executable, '-m', 'interpreter.train',
                     '--config', 'configs/candidate_interpreter.json',
                     '--run-id', 'interp_slice']).returncode
assert rc == 0, 'train failed (read the [interp] output above)'
ADAPTER = sorted(glob.glob('models/interpreter/interp_slice_*'))[-1]
print('trained:', ADAPTER)

trained: models/interpreter/interp_slice_smokefull


In [ ]:
# CELL 4 - score on the untouched holdout; read METRIC + the GO/NO-GO components
import subprocess, sys, glob
ADAPTER = sorted(glob.glob('models/interpreter/interp_slice_*'))[-1]
subprocess.run([sys.executable, '-m', 'interpreter.score',
                '--config', 'configs/candidate_interpreter.json', '--adapter', ADAPTER])
print('\nGO/NO-GO: METRIC (unit-macro joint) and route_accuracy must beat a trivial always-grade\n'
      'baseline, and attribute_f1[real_lut] must be > 0. If not -> the approach is wrong; stop\n'
      'before the full ~2761-LUT caption run.')

In [15]:
!cd /content/SLM && python -m interpreter.score \
  --config configs/candidate_interpreter.json \
  --adapter $(ls -d models/interpreter/interp_slice_* | tail -1) 2>&1 | tail -40

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 16958.26it/s]
[interp] score_summary
{"score_summary": {"attribute_f1": {"overall": {"mean": 0.08461405827704621, "n": 85}, "procedural": null, "real_lut": {"mean": 0.08461405827704621, "n": 85}}, "attribute_f1_by_style": {"concept": {"mean": 0.061341692213949583, "n": 17}, "literal": {"mean": 0.12464531124131162, "n": 17}, "metaphor": {"mean": 0.044134697357203746, "n": 17}, "mood": {"mean": 0.12653270003653636, "n": 17}, "slang": {"mean": 0.06641589053622975, "n": 17}}, "interpreter_over_refusal_rate": {"high": 0.4827009216378103, "n": 85, "rate": 0.3764705882352941}, "metric": 0.6798851947921363, "metric_ci": [0.5261564013192657, 0.8172962531373779], "n": 108, "n_units": 34, "parse_ok_rate": 0.7129629629629629, "per_route_recall": {"clarify": {"low": 0.0, "n": 1, "recall": 0.0}, "grade": {"low": 0.5172990783621897, "n": 85, "recall": 0.6235294117647059}, "refuse": {"low": 0.8513451253913791, "n": 22, "recall": 1.0}}, "refuse_ki

In [17]:
# 1. pull the fix
!cd /content/SLM && git fetch origin feat/two-stage -q && git reset --hard origin/feat/two-stage -q && git log --oneline -1

# 2. drop the duplicate clarify rows from the cache (keep gamut)
import json
p = "/content/SLM/data/active_sft/route_supplement_cache.jsonl"
kept = [l for l in open(p) if l.strip() and not json.loads(l).get("id","").startswith("unsup_clarify_")]
open(p, "w").writelines(kept)
print("kept", len(kept), "non-clarify cache rows (gamut stays)")

# 3. regenerate ONLY clarify (gamut is skipped as already-done) + rebuild the corpus
!cd /content/SLM && python -m scripts.generate_route_supplement --n-clarify 150 --n-gamut 150 && \
python -m scripts.build_interpreter_corpus

# 4. diversity check
import collections
rows = [json.loads(l) for l in open(p) if l.strip()]
clar = [r["prompt"] for r in rows if r.get("id","").startswith("unsup_clarify_") and r.get("status")=="generated"]
print("clarify generated:", len(clar), "| DISTINCT:", len(set(clar)))
for s in list(dict.fromkeys(clar))[:8]: print("  -", s)

a471f1a (HEAD -> feat/two-stage, origin/feat/two-stage) Fix clarify diversity collapse: per-item framing seeds
kept 150 non-clarify cache rows (gamut stays)
[supplement] resuming: 150 rows already in data/active_sft/route_supplement_cache.jsonl
  unsup_clarify_000001: generated  touch up the tones
  unsup_clarify_000002: generated  Can you do something with the overall feel of this photo so it looks m
  unsup_clarify_000003: generated  idk something feels off about this pic, can u just tweak it a bit?
  unsup_clarify_000004: generated  Could you please adjust the overall color and tone of this image so th
  unsup_clarify_000005: generated  Something about the color in this photo just feels wrong, can you sort
  unsup_clarify_000006: generated  something feels off about the vibe of this photo, can you just fix it 
  unsup_clarify_000007: generated  can you touch up the tones so this photo feels more Instagram-worthy?
  unsup_clarify_000008: generated  I need to present this in my client

In [18]:
!cd /content/SLM && python -m interpreter.train --config configs/candidate_interpreter.json --run-id interp_slice 2>&1 | tail -15
!cd /content/SLM && python -m interpreter.score --config configs/candidate_interpreter.json --adapter $(ls -d models/interpreter/interp_slice_* | tail -1) 2>&1 | tail -40

[interp] epoch 2 optim_step 240/465 loss 0.5790
[interp] epoch 2 optim_step 260/465 loss 0.5105
[interp] epoch 3 optim_step 280/465 loss 0.5443
[interp] epoch 3 optim_step 300/465 loss 0.4606
[interp] epoch 3 optim_step 320/465 loss 0.5229
[interp] epoch 3 optim_step 340/465 loss 0.4235
[interp] epoch 3 optim_step 360/465 loss 0.4984
[interp] epoch 4 optim_step 380/465 loss 0.5808
[interp] epoch 4 optim_step 400/465 loss 0.5427
[interp] epoch 4 optim_step 420/465 loss 0.5230
[interp] epoch 4 optim_step 440/465 loss 0.5186
[interp] epoch 4 optim_step 460/465 loss 0.5484
Writing model shards: 100%|██████████| 1/1 [00:16<00:00, 16.13s/it]
[interp][OK] {'rows_trained': 2954, 'optim_steps': 461, 'mean_loss': 0.634841135166351, 'base_model_id': 'Qwen/Qwen2.5-0.5B-Instruct', 'tuning_mode': 'full', 'adapter': 'models/interpreter/interp_slice_smokefull', 'status': 'ok'}
{"interpreter_summary": {"rows_trained": 2954, "optim_steps": 461, "mean_loss": 0.634841135166351, "base_model_id": "Qwen/Qwen